# Recource Counting

## 引例

- 问题：用1024张H100，在15万亿个token上，训练一个70B参数的模型，需要多长时间？
- 计算：
    - total_flops = 6 * 70e9 * 15e12
    - h100_flop/sec = 1979e12 / 2
    - mfu = 0.5  **?**
    - flops/day = h100_flop/sec * mfu * 1024 *60 *60 * 24
    - days = total_flops * flops/day

- 问题：如果用八块H100配合adamW，能训练的最大模型有多大？
- 计算：
    - h100_bytes = 80e9(80GB)
    - bytes/parameter = 2 + 2 + (4 + 4)
    - num_parameters = h100_bytes * 8 / bytes/parameter
    

本节要点：
- pytorch
- 写代码养成资源核算习惯

## 从头搭建

什么是tensors(张量)：
- 参数，梯度，优化器状态，激活值
- 模型本质是一堆形状和精度各异的张量

**float32**（单精度）
- 1位：符号位
- 8位：指数位



In [8]:
# import torch.nn.functional as F
import torch
from torch import nn
from pathlib import Path
import sys
sys.path.insert(0, r".\lectures-main\lectures-main")


from gpu_util import cuda_if_available, get_max_memory_usage


In [5]:
x = torch.zeros(4,8)
x.dtype


torch.float32

In [ ]:
x.numel() # .numel()方法计算元素总个数

32

In [11]:
x.element_size() # 浮点数4bytes = 32 bits

4

In [12]:
get_memory_usage(x)

128

In [ ]:
get_memory_usage(torch.empty(12288 * 4, 12288))

2415919104

In [9]:
def get_memory_usage(x:torch.Tensor):
    return x.numel() * x.element_size()